# 수업 당시 코드 참고본

과제 이후 포트폴리오 정리 과정에서 출력과 개인 PC 경로를 제거한 참고본입니다. 원본 계산의 문제도 기록으로 남아 있습니다. 재실행은 `src/fomo_eda.py`를 사용하세요. 수정 사항은 `docs/review_status.md`에 설명했습니다.


# 페키지 로드 및 데이터 구조 확인

In [ ]:
!pip install pandas openpyxl

In [ ]:
# 페키지 로드
import os
import pandas as pd
import numpy as np
import openpyxl as op

In [ ]:
import pandas as pd

# ---------------------------------------------------
# 1. 데이터 경로 설정
# ---------------------------------------------------
DATA_DIR = r"<LOCAL_PATH>"

stock_path  = fr"{DATA_DIR}\주식통합.xlsx"
weight_path = fr"{DATA_DIR}\빅카인즈 검색, 뉴스_가중치.xlsx"
trend_path  = fr"{DATA_DIR}\뉴스, 검색 데이터.xlsx"

print("주식통합 :", stock_path)
print("가중치 파일 :", weight_path)
print("뉴스/검색 :", trend_path)

# ---------------------------------------------------
# 2. 데이터 로드
# ---------------------------------------------------
df_stock = pd.read_excel(stock_path)
df_weight = pd.read_excel(weight_path)
df_trend = pd.read_excel(trend_path)

# ---------------------------------------------------
# 3. 기본 정보 출력
# ---------------------------------------------------
print("\n[주식통합 데이터 head]")
print(df_stock.head())
print("\n[주식통합 info]")
print(df_stock.info())

print("\n[가중치 데이터 head]")
print(df_weight.head())
print("\n[가중치 데이터 info]")
print(df_stock.info())
print("\n[뉴스/검색 데이터 head]")
print(df_trend.head())
print("\n[뉴스/검색 데이터 info]")
print(df_stock.info())


# 데이터 전처리 (날짜 변환, 컬럼명, Wide->long)

In [ ]:
# 주식 데이터 날짜 변환(object -> datetime64)
df_stock['date'] = pd.to_datetime(df_stock['일자'], format='%Y/%m/%d')

# 필요하면 원본 '일자'는 드롭
df_stock = df_stock.drop(columns=['일자'])

print(df_stock[['종목', 'date']].head())
print(df_stock.info())


In [ ]:
# 뉴스/검색 데이터 날짜 변환(object -> datetime64)
# -----------------------------
df_trend['date'] = pd.to_datetime(df_trend['일자'])

df_trend = df_trend.drop(columns=['일자'])

print(df_trend[['date']].head())
print(df_trend.info())


In [ ]:
# 검색량 컬럼 찾기 (문자열에 '검색량' 포함된 것)
search_cols = [c for c in df_trend.columns if '검색량' in c]

print("검색량 컬럼들:")
print(search_cols)

In [ ]:
# 뉴스 건수 컬럼 찾기
#  - 'NEWS 건수_' 로 시작하거나
#  - '건수_' 로 시작하는 컬럼 포함
# ------------------------------------
news_cols = [c for c in df_trend.columns 
             if c.startswith('NEWS 건수_') or c.startswith('건수_')]

print("뉴스 건수 컬럼들:")
print(news_cols)

In [ ]:
# 검색량 wide → long 변환
search_long = df_trend.melt(
    id_vars=['date'],           # 기준 축
    value_vars=search_cols,     # 펼칠 컬럼들
    var_name='검색컬럼명',
    value_name='검색량'
)

# 컬럼명에서 종목명 추출 (예: 'NAVER 검색량_펄어비스' → '펄어비스')
search_long['종목'] = search_long['검색컬럼명'].str.split('_').str[-1]

# 필요없는 원래 컬럼명은 제거
search_long = search_long.drop(columns=['검색컬럼명'])

print(search_long.head(10), search_long.tail(10))


In [ ]:
# 뉴스 건수 wide → long 변환
# ------------------------------------
news_long = df_trend.melt(
    id_vars=['date'],
    value_vars=news_cols,
    var_name='뉴스컬럼명',
    value_name='뉴스건수'
)

# 'NEWS 건수_삼성전자' → '삼성전자', '건수_셀트리온' → '셀트리온'
news_long['종목'] = news_long['뉴스컬럼명'].str.split('_').str[-1]

news_long = news_long.drop(columns=['뉴스컬럼명'])

print(news_long.head(10), search_long.tail(10))


In [ ]:
# 검색량 + 뉴스건수 병합 (date + 종목 기준)
# ------------------------------------
issue_long = pd.merge(
    search_long,
    news_long,
    on=['date', '종목'],
    how='outer'   # 어떤 쪽이 비어 있어도 다 살려두기
)

issue_long = issue_long[['date', '종목', '검색량', '뉴스건수']]

print(issue_long.head(20))
print(issue_long.info())


In [ ]:
df_stock_issue = df_stock.merge(
    issue_long,
    on=['종목', 'date'],
    how='left'
)

In [ ]:
#(data dtype 다시 확인.)
print(df_stock.info())

# 데이터 병합

In [ ]:
import numpy as np
import pandas as pd

# 2) 종목 필터: 이슈 데이터에 있는 종목만 사용
issue_stocks = issue_long['종목'].unique()
df_stock = df_stock[df_stock['종목'].isin(issue_stocks)].copy()

# 3) 날짜 범위도 이슈 데이터 범위 안으로 자르기
start_date = issue_long['date'].min()
end_date   = issue_long['date'].max()

df_stock = df_stock[
    (df_stock['date'] >= start_date) &
    (df_stock['date'] <= end_date)
].copy()

# 4) 정렬
df_stock = df_stock.sort_values(['종목', 'date']).reset_index(drop=True)

print("주식 종목:", sorted(df_stock['종목'].unique()))
print("주식 날짜 범위:", df_stock['date'].min(), "~", df_stock['date'].max())


In [ ]:
# 이슈 데이터도 정렬
df_issue = issue_long.sort_values(['종목', 'date']).reset_index(drop=True)

print(df_issue.head())
print(df_issue.info())

In [ ]:
# 이슈 기준으로 주식데이터 merge
df_all = df_issue.merge(
    df_stock,
    on=['종목', 'date'],
    how='left'   # 이슈 기준으로 모두 살리고, 주식 없는 날은 NaN
)

print(df_all.head(20))
print(df_all[['종목', 'date', '검색량', '뉴스건수', '종가', '거래량']].head(30))
print(df_all.info())

In [ ]:
# 변수 타임별로 ffill / 0 처리 분리 적용
# price/보유량(state) -> ffill(주말에도 상태 유지)
# price(상태) 계열 -> ffill금지, NaN이면 0처리
# 4-1. 컬럼 그룹 나누기
#----------------------------------------------------------
# price(상태) 계열
price_cols = ['종가', '시가', '고가', '저가']

# 보유량(상태) 계열
hold_cols = ['시가총액', '상장주식수', '외국인 보유수량', '외국인 지분율',
             '외국인 한도수량', '외국인 한도소진율']

# flow(흐름) 계열: 거래가 있는 날에만 의미가 있는 값들
flow_cols = ['거래량', '거래대금',
             '대비', '등락률',
             '개인(매도)', '개인(매수)', '개인(순매수)']


In [ ]:
# 4-3. price + holding만 ffill
# 종목별로 price/holding 상태를 주말까지 이어줌
df_all[price_cols + hold_cols] = (
    df_all
    .groupby('종목')[price_cols + hold_cols]
    .ffill()
)


In [ ]:
# 4-3. flow 계열은 ffill 금지 → NaN이면 0
df_all[flow_cols] = df_all[flow_cols].fillna(0)

In [ ]:
print(df_all.head(20))

In [ ]:
# 파생변수 : 이슈강도, 변동성, 로그 거래량

# 정렬 한 번 더 (안전하게)
df_all = df_all.sort_values(['종목', 'date']).reset_index(drop=True)

# 5-1. 이슈강도: 우선 뉴스건수 그대로 사용 (1차 버전)
df_all["뉴스_log"] = np.log1p(df_all["뉴스건수"])
df_all["검색_log"] = np.log1p(df_all["검색량"])

df_all["이슈강도"] = df_all["뉴스_log"] + df_all["검색_log"]



# 5-2. 변동성 지표: 등락률 절댓값
df_all['등락률_abs'] = df_all['등락률'].abs()

# 5-3. 거래량 로그변환 (0도 있을 수 있으니 log1p)
df_all['거래량_log'] = np.log1p(df_all['거래량'])

print("▶ 파생변수 예시")
print(df_all[['종목','date','이슈강도','등락률','등락률_abs','거래량','거래량_log']].head(20))


In [ ]:
# 6-1. 종목별 거래일 + next_trade_date는 그대로 OK
trade_dates = (
    df_stock[['종목','date']]
    .drop_duplicates()
    .sort_values(['종목','date'])
    .copy()
)
trade_dates['next_trade_date'] = trade_dates.groupby('종목')['date'].shift(-1)

print(trade_dates.head(10))


In [ ]:
# 혹시 모를 dtype 문제 대비
df_all['date'] = pd.to_datetime(df_all['date'])
trade_dates['date'] = pd.to_datetime(trade_dates['date'])

result_list = []

for stock, g in df_all.groupby('종목'):
    # 1) 이 종목의 이슈 타임라인 (주말 포함 가능)
    g = g.sort_values('date').reset_index(drop=True)
    
    # 2) 이 종목의 거래일 + next_trade_date
    td = trade_dates[trade_dates['종목'] == stock].copy()
    td = td.sort_values('date').reset_index(drop=True)
    
    # 3) asof를 위해 컬럼명 맞추기 (right의 키 이름 바꿔줌)
    td_renamed = td.rename(columns={'date': 'trade_date'})
    
    # 4) 이 종목의 각 날짜(date)에 대해
    #    - trade_date <= date 인 마지막 거래일을 찾고
    #    - 그 거래일의 next_trade_date를 붙임
    merged = pd.merge_asof(
        g,
        td_renamed[['trade_date', 'next_trade_date']],
        left_on='date',
        right_on='trade_date',
        direction='backward'
    )
    
    # trade_date 컬럼은 이제 필요 없으니 삭제
    merged = merged.drop(columns=['trade_date'])
    
    result_list.append(merged)

# 모든 종목 다시 합치기
df_all_with_next = (
    pd.concat(result_list, axis=0)
    .sort_values(['종목', 'date'])
    .reset_index(drop=True)
)

print("▶ 종목별 asof 이후 예시")
print(df_all_with_next[['종목','date','next_trade_date']].head(30))


In [ ]:
# 다음 거래일의 등락률 / 거래량_log 테이블 준비
future = (
    df_all_with_next[['종목','date','등락률','거래량_log']]
    .rename(columns={
        'date': 'next_trade_date',
        '등락률': '등락률_next',
        '거래량_log': '거래량_log_next',
    })
)

# next_trade_date 기준으로 merge
df_all_final = df_all_with_next.merge(
    future,
    on=['종목','next_trade_date'],
    how='left'
)

print("▶ 다음 거래일 반응 컬럼 확인")
print(
    df_all_final[
        ['종목','date','이슈강도','next_trade_date',
         '등락률','등락률_next','거래량_log','거래량_log_next']
    ].head(40)
)

# 결측치 확인 및 처리

In [ ]:
df_all_final.isna().sum()

next_trade_date, 등락률_next, 거래량_log_next은 다음 거래일의 데이터이다.
여기서 데이터상 거래가 끝나는 2025-11-28 뒤의 거래는 없기에 NaN으로 뜨는 것, 0으로 처리하면 오해가 생길 우려가 있으니 결측치 처리는 생략.

In [ ]:
output_path = r"<LOCAL_PATH>"
df_all_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print("저장 완료:", output_path)

# 분석

In [ ]:
!pip install seaborn

In [ ]:
!pip install scikit-learn

In [ ]:
# 글꼴 설정
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc

# 맑은 고딕 경로 등록
font_path = "C:/Windows/Fonts/malgun.ttf"   # 경로 그대로 사용!
font = font_manager.FontProperties(fname=font_path).get_name()

# 폰트 설정
rc('font', family=font)

# 마이너스 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 1.클러스터링 분석

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1) 군집용 feature 추출
# -------------------------------

features = []

for stock, sub in df_all_final.groupby("종목"):
    
    sub = sub.sort_values("date")

    # 이슈강도 평균
    mean_issue = sub["이슈강도"].mean()

    # 검색량 평균
    mean_search = sub["검색량"].mean()

    # 뉴스건수 평균
    mean_news = sub["뉴스건수"].mean()

    # 이슈강도 ↔ 다음날 거래량 상관관계
    corr_issue_vol = sub["이슈강도"].corr(sub["거래량_log_next"])

    # 이슈강도 ↔ 다음날 등락률 상관관계
    corr_issue_ret = sub["이슈강도"].corr(sub["등락률_next"])

    # 종목별 평균 다음날 거래량 변화 정도
    mean_vol_next = sub["거래량_log_next"].mean()

    # 종목별 평균 다음날 등락률
    mean_ret_next = sub["등락률_next"].mean()

    features.append([
        stock,
        mean_issue,
        mean_search,
        mean_news,
        corr_issue_vol,
        corr_issue_ret,
        mean_vol_next,
        mean_ret_next
    ])

# DataFrame 생성
feature_df = pd.DataFrame(features, columns=[
    "종목",
    "mean_issue",
    "mean_search",
    "mean_news",
    "corr_issue_volume",
    "corr_issue_return",
    "mean_volume_next",
    "mean_return_next"
])

feature_df = feature_df.set_index("종목")

# NaN → 0 처리 (상관계수 NaN 방지용)
feature_df = feature_df.fillna(0)

print(feature_df)

In [ ]:
from sklearn.preprocessing import StandardScaler

X = feature_df.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# -------------------------------------------------------------
# 종목별 이슈강도 Top1 날짜 + 등락률/거래량 변화 (직전 '거래일' 기준)
# -------------------------------------------------------------

top1_result = {}

for stock in df_all["종목"].unique():
    # 해당 종목만 추출 후 날짜 기준 정렬
    temp = df_all[df_all["종목"] == stock].sort_values("date").copy()

    # 1) 이슈강도 Top1 날짜 찾기 (날짜별 평균 이슈강도 기준)
    top1 = (
        temp.groupby("date")["이슈강도"]
        .mean()
        .reset_index()
        .sort_values("이슈강도", ascending=False)
        .head(1)
    )

    row = top1.iloc[0]
    d = row["date"]          # Top1 이슈 날짜
    issue_score = row["이슈강도"]

    # 2) 해당 날짜 행 (today)
    today = temp[temp["date"] == d].iloc[0]

    # 3) '직전 거래일' 찾기: 날짜 < d 이면서 거래량 > 0 인 마지막 날
    prev_trade = temp[(temp["date"] < d) & (temp["거래량"] > 0)].tail(1)

    if len(prev_trade) > 0:
        y = prev_trade.iloc[0]

        # 등락률 변화
        rate_change = today["등락률"] - y["등락률"]

        # 거래량 변화율 (%)
        volume_change = (today["거래량"] - y["거래량"]) / y["거래량"] * 100

        prev_date = y["date"]
        prev_volume = y["거래량"]
        prev_return = y["등락률"]

    else:
        # 이전 거래일 자체가 없으면 변화 계산 불가
        rate_change = None
        volume_change = None
        prev_date = None
        prev_volume = None
        prev_return = None

    top1_result[stock] = {
        "이슈 Top1 날짜": d,
        "이슈강도": issue_score,
        "현재 등락률": today["등락률"],
        "직전거래일 등락률": prev_return,
        "전일대비 등락률 변화": rate_change,
        "현재 거래량": today["거래량"],
        "직전거래일": prev_date,
        "직전거래일 거래량": prev_volume,
        "전일대비 거래량 변화(%)": volume_change,
    }

# 보기 좋게 DataFrame으로 변환
top1_df = pd.DataFrame.from_dict(top1_result, orient="index")
print(top1_df)



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# df_all : 전체 종목-일자 데이터
# top1_df : 종목별 이슈 Top1 날짜 포함 (index=종목명)

result_list = []

for stock in top1_df.index:
    top_date = pd.to_datetime(top1_df.loc[stock, "이슈 Top1 날짜"])

    temp = df_all[df_all["종목"] == stock].copy()
    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp.sort_values("date").reset_index(drop=True)

    # 이벤트일 인덱스 찾기
    idx_list = temp.index[temp["date"] == top_date].tolist()
    if not idx_list:
        continue
    idx = idx_list[0]

    # ±7 거래일 구간
    start_idx = max(0, idx - 7)
    end_idx   = min(len(temp)-1, idx + 7)
    window = temp.iloc[start_idx:end_idx+1].copy()

    # 1) 수익률 변동성(표준편차)
    ret_std = window["등락률"].std()

    # 2) 거래량 스파이크 : 이벤트일 거래량 / 직전 20일 평균
    past_20 = temp.iloc[max(0, idx-20):idx]
    if len(past_20) > 0:
        vol_spike = temp.loc[idx, "거래량"] / past_20["거래량"].mean()
    else:
        vol_spike = np.nan

    result_list.append([stock, ret_std, vol_spike])

# 통합 DF 생성
cluster_df = pd.DataFrame(result_list, columns=["종목", "수익률_변동성", "거래량_스파이크"])


In [ ]:
# 결측 제거
cluster_df = cluster_df.dropna()

# 표준화
scaler = StandardScaler()
cluster_df[["ret_z", "vol_z"]] = scaler.fit_transform(
    cluster_df[["수익률_변동성", "거래량_스파이크"]]
)

# 민감도 점수 계산 (핵심!)
cluster_df["이슈_민감도"] = cluster_df["ret_z"] + cluster_df["vol_z"]


In [ ]:
# 최적의 k값 찾기
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

inertias, sil_scores = [], []
K_range = range(2, 7)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Elbow
plt.figure(figsize=(8,4))
plt.plot(K_range, inertias, marker='o')
plt.title("Elbow Method")
plt.xlabel("k"); plt.ylabel("Inertia")
plt.grid(True)
plt.show()

# Silhouette
plt.figure(figsize=(8,4))
plt.plot(K_range, sil_scores, marker='o')
plt.title("Silhouette Score")
plt.xlabel("k"); plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

In [ ]:
# K=3
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_df["cluster"] = kmeans.fit_predict(
    cluster_df[["ret_z", "vol_z"]]
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.scatterplot(
    data=cluster_df,
    x="ret_z", y="vol_z",
    hue="cluster", s=200
)

for i, row in cluster_df.iterrows():
    plt.text(row["ret_z"]+0.03, row["vol_z"]+0.03, row["종목"], fontsize=12)

plt.xlabel("수익률 변동성 (표준화)")
plt.ylabel("거래량 스파이크 (표준화)")
plt.title("이슈 민감도 기반 K-means 클러스터링")
plt.axhline(0, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.show()


In [ ]:
pip install scikit-learn

# 2.종목별 시계열 흐름 

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ================================
# 1) 스파이크 감지 함수
# ================================
def detect_spike(series, window=20, std_k=2):
    s = series.fillna(0)
    roll_mean = s.rolling(window, min_periods=5).mean()
    roll_std  = s.rolling(window, min_periods=5).std()
    spike = (s > roll_mean + std_k * roll_std).astype(int)
    return spike.fillna(0)


# ================================
# 2) 시계열 시각화 (상위 퍼센타일 스파이크만 표시)
# ================================
def plot_flow_shading(df, stock, percentile=0.90):

    temp = df[df["종목"] == stock].sort_values("date").copy()
    temp = temp[["date", "검색량", "뉴스건수", "개인(순매수)", "종가"]].fillna(0)

    # ------ 스파이크 감지 ------
    temp["spike_search"] = detect_spike(temp["검색량"])
    temp["spike_price"] = detect_spike(temp["종가"])

    # 하나라도 spike이면 스파이크로 판단
    temp["spike"] = ((temp["spike_search"] == 1) | (temp["spike_price"] == 1)).astype(int)

    # ------ 정규화 ------
    scaler = MinMaxScaler()
    cols = ["검색량", "뉴스건수", "개인(순매수)", "종가"]
    temp[[c + "_s" for c in cols]] = scaler.fit_transform(temp[cols])

    # ------ 스파이크 중 상위 퍼센타일만 선택 ------
    spike_df = temp[temp["spike"] == 1].copy()
    if len(spike_df) > 0:
        threshold = spike_df["검색량_s"].quantile(percentile)
        strong_spike_df = spike_df[spike_df["검색량_s"] >= threshold]
    else:
        strong_spike_df = pd.DataFrame()

    # ------ 플롯 ------
    plt.figure(figsize=(20,6))

    plt.plot(temp["date"], temp["검색량_s"], label="검색량", color="blue", linewidth=1.8)
    plt.plot(temp["date"], temp["뉴스건수_s"], label="뉴스건수", color="orange", linewidth=1.8)
    plt.plot(temp["date"], temp["개인(순매수)_s"], label="개인 순매수", color="green", linewidth=1.8)
    plt.plot(temp["date"], temp["종가_s"], label="주가", color="red", linewidth=1.8)

    # ------ 강한 스파이크만 표시 ------
    for d in strong_spike_df["date"]:
        plt.axvspan(d, d + pd.Timedelta(days=1), color="red", alpha=0.15)

    for _, row in strong_spike_df.iterrows():
        x = row["date"]
        y = max(row["검색량_s"], row["종가_s"]) + 0.15

        plt.scatter(x, y, color="red", s=90, zorder=5)
        plt.text(x, y + 0.03, "SPK", fontsize=9, color="red", weight="bold")

    plt.title(f"[{stock}] 검색량·주가 스파이크 기반 시계열 흐름 (상위 퍼센타일 표시)", fontsize=17)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for s in df_all_final["종목"].unique():
    plot_flow_shading(df_all_final, s)

이슈 반응

In [ ]:
# 이슈일 정의
# 해당 종목의 이슈강도 > 해당 종목의 rolling 평균 + 2 * 표준편차

In [ ]:
# 1) 종목별 rolling 평균 & 표준편차 계산
# -----------------------------------------------------
df_all['이슈_mean'] = df_all.groupby("종목")['이슈강도'].transform(lambda x: x.rolling(20, min_periods=5).mean())
df_all['이슈_std']  = df_all.groupby("종목")['이슈강도'].transform(lambda x: x.rolling(20, min_periods=5).std())

# -----------------------------------------------------
# 2) 이벤트일(Event Day) 정의
# -----------------------------------------------------
df_all['이벤트여부'] = (df_all['이슈강도'] > df_all['이슈_mean'] + 2 * df_all['이슈_std']).astype(int)

print("▶ 이벤트 발생 예시")
print(df_all[df_all['이벤트여부'] == 1].head(10)[['종목','date','이슈강도','이슈_mean','이슈_std']])

In [ ]:
# 3) 이벤트 스터디: ±3일 변동 추출
# -----------------------------------------------------
event_window = []

for stock, sub in df_all.groupby("종목"):
    sub = sub.sort_values("date").reset_index(drop=True)

    event_idx = sub[sub['이벤트여부'] == 1].index

    for idx in event_idx:
        for lag in range(-3, 4):   # -3 ~ +3일
            new_idx = idx + lag
            if 0 <= new_idx < len(sub):
                event_window.append({
                    "종목": stock,
                    "event_date": sub.loc[idx, "date"],
                    "lag": lag,
                    "등락률": sub.loc[new_idx, "등락률"],
                    "거래량_log": sub.loc[new_idx, "거래량_log"],
                    "이슈강도": sub.loc[new_idx, "이슈강도"]
                })

event_df = pd.DataFrame(event_window)

print("▶ 이벤트 윈도우 예시")
print(event_df.head(15))


In [ ]:
# 4) lag별 평균 변화 계산
# -----------------------------------------------------
lag_result = event_df.groupby("lag")[["등락률", "거래량_log"]].mean().reset_index()

print("▶ lag별 평균 등락률/거래량 변화")
print(lag_result)

In [ ]:
# 등락률 이벤터 스터디 플롯
plt.figure(figsize=(10,5))
sns.lineplot(data=lag_result, x="lag", y="등락률", marker="o")
plt.axvline(0, color='red', linestyle='--')
plt.title("이슈 이벤트 전후 등락률 변화(±3일)")
plt.xlabel("이벤트 기준일 (lag)")
plt.ylabel("평균 등락률 (%)")
plt.grid(True)
plt.show()

# 거래량 이벤트 스터디 플롯
plt.figure(figsize=(10,5))
sns.lineplot(data=lag_result, x="lag", y="거래량_log", marker="o")
plt.axvline(0, color='red', linestyle='--')
plt.title("이슈 이벤트 전후 거래량 변화(±3일)")
plt.xlabel("이벤트 기준일 (lag)")
plt.ylabel("평균 거래량 log")
plt.grid(True)
plt.show()


In [ ]:
# 종목별 Top 이벤트일(이슈강도 높은 날)

# 종목 리스트
stock_list = df_all['종목'].unique()

# 각 종목별 top 이벤트(이슈강도 상위 10개)
event_days_by_stock = {}

for s in stock_list:
    temp = df_all[df_all['종목'] == s].sort_values('이슈강도', ascending=False)
    event_days_by_stock[s] = temp['date'].head(10).tolist()

event_days_by_stock


In [ ]:
# 종목별 이벤트 전후 ±3일 데이터 추출 함수
def get_event_window(df, stock, event_dates, window=3):
    frames = []
    df_s = df[df['종목'] == stock]

    for d in event_dates:
        for lag in range(-window, window+1):
            ref_date = d + pd.Timedelta(days=lag)
            row = df_s[df_s['date'] == ref_date]
            if len(row) > 0:
                frames.append({
                    '종목': stock,
                    'event_date': d,
                    'lag': lag,
                    '등락률': row['등락률'].values[0],
                    '거래량_log': row['거래량_log'].values[0]
                })

    return pd.DataFrame(frames)


In [ ]:
# 전체 종목별 이벤트 윈도우 생성
event_windows = []

for s in stock_list:
    ev = get_event_window(df_all, s, event_days_by_stock[s], window=3)
    event_windows.append(ev)

event_windows = pd.concat(event_windows, ignore_index=True)


In [ ]:
# 종목별 평균 패턴 계산
event_mean_by_stock = (
    event_windows
    .groupby(['종목', 'lag'])
    .agg({'등락률':'mean', '거래량_log':'mean'})
    .reset_index()
)


In [ ]:
# 종목별 이벤트 스터디 그래프
import matplotlib.pyplot as plt

for s in stock_list:
    temp = event_mean_by_stock[event_mean_by_stock['종목'] == s]

    fig, axes = plt.subplots(2, 1, figsize=(12,10))

    # 1) 등락률
    axes[0].plot(temp['lag'], temp['등락률'], marker='o')
    axes[0].axvline(0, color='red', linestyle='--')
    axes[0].set_title(f"[{s}] 이슈 전후 등락률 변화(±3일)")
    axes[0].set_xlabel("lag")
    axes[0].set_ylabel("등락률(%)")

    # 2) 거래량
    axes[1].plot(temp['lag'], temp['거래량_log'], marker='o', color='orange')
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].set_title(f"[{s}] 이슈 전후 거래량 변화(±3일)")
    axes[1].set_xlabel("lag")
    axes[1].set_ylabel("거래량_log")

    plt.tight_layout()
    plt.show()


In [ ]:
# -------------------------------------------------------------
# 종목별 이슈강도 Top3 날짜 추출
# -------------------------------------------------------------

stock_top3 = {}

for stock in df_all["종목"].unique():
    temp = df_all[df_all["종목"] == stock]

    # 날짜별 이슈강도(이미 합쳐진 값: 검색량 + 뉴스건수)
    daily_issue = temp.groupby("date")["이슈강도"].mean().reset_index()

    # Top3 추출
    top3 = daily_issue.sort_values("이슈강도", ascending=False).head(1)

    stock_top3[stock] = top3

# 출력
for stock, table in stock_top3.items():
    print(f"\n📌 [{stock}] 이슈강도 Top 날짜")
    print(table)

In [ ]:
# -------------------------------------------------------------
# 종목별 이슈강도 Top1 날짜 + 등락률/거래량 변화 (직전 '거래일' 기준)
# -------------------------------------------------------------

top1_result = {}

for stock in df_all["종목"].unique():
    # 해당 종목만 추출 후 날짜 기준 정렬
    temp = df_all[df_all["종목"] == stock].sort_values("date").copy()

    # 1) 이슈강도 Top1 날짜 찾기 (날짜별 평균 이슈강도 기준)
    top1 = (
        temp.groupby("date")["이슈강도"]
        .mean()
        .reset_index()
        .sort_values("이슈강도", ascending=False)
        .head(1)
    )

    row = top1.iloc[0]
    d = row["date"]          # Top1 이슈 날짜
    issue_score = row["이슈강도"]

    # 2) 해당 날짜 행 (today)
    today = temp[temp["date"] == d].iloc[0]

    # 3) '직전 거래일' 찾기: 날짜 < d 이면서 거래량 > 0 인 마지막 날
    prev_trade = temp[(temp["date"] < d) & (temp["거래량"] > 0)].tail(1)

    if len(prev_trade) > 0:
        y = prev_trade.iloc[0]

        # 등락률 변화
        rate_change = today["등락률"] - y["등락률"]

        # 거래량 변화율 (%)
        volume_change = (today["거래량"] - y["거래량"]) / y["거래량"] * 100

        prev_date = y["date"]
        prev_volume = y["거래량"]
        prev_return = y["등락률"]

    else:
        # 이전 거래일 자체가 없으면 변화 계산 불가
        rate_change = None
        volume_change = None
        prev_date = None
        prev_volume = None
        prev_return = None

    top1_result[stock] = {
        "이슈 Top1 날짜": d,
        "이슈강도": issue_score,
        "현재 등락률": today["등락률"],
        "직전거래일 등락률": prev_return,
        "전일대비 등락률 변화": rate_change,
        "현재 거래량": today["거래량"],
        "직전거래일": prev_date,
        "직전거래일 거래량": prev_volume,
        "전일대비 거래량 변화(%)": volume_change,
    }

# 보기 좋게 DataFrame으로 변환
top1_df = pd.DataFrame.from_dict(top1_result, orient="index")
print(top1_df)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set(style="whitegrid", font="Malgun Gothic")

def plot_event_reaction(stock):
    print(f"\n===== [{stock}] 이벤트 플롯 시작 =====")

    # 1) Top1 날짜 가져오기
    if stock not in top1_df.index:
        print(f"[경고] top1_df에 '{stock}' 이(가) 없습니다.")
        return
    
    top_date = pd.to_datetime(top1_df.loc[stock, "이슈 Top1 날짜"])

    # 2) 종목 데이터
    temp = df_all[df_all["종목"] == stock].copy()
    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp.sort_values("date").reset_index(drop=True)

    # 3) 거래일 기준 ±7일 인덱싱
    idx_list = temp.index[temp["date"] == top_date].tolist()
    if not idx_list:
        print("[경고] 이 날짜는 실제 거래일이 아님(휴일 등)")
        return
    idx = idx_list[0]

    start_idx = max(0, idx - 7)
    end_idx   = min(len(temp) - 1, idx + 7)

    window = temp.iloc[start_idx:end_idx + 1].copy()
    window["lag"] = (window["date"] - top_date).dt.days

    # ===== 핵심! 등락률(%) 그대로 사용, Δ등락률 계산 =====
    window["Δ등락률"] = window["등락률"].diff()

    # 거래량 log
    window["거래량_log"] = np.log1p(window["거래량"])

    # ---------------------------------------------------------
    # 4) 🔥 Y축 고정: 등락률은 -20% ~ +20%로 통일
    # ---------------------------------------------------------
    Y_MIN, Y_MAX = -20, 20

    # 5) 플롯
    fig, ax1 = plt.subplots(figsize=(13, 5))

    sns.lineplot(ax=ax1, data=window, x="date", y="등락률",
                 marker="o", label="수익률(%)")
    sns.lineplot(ax=ax1, data=window, x="date", y="Δ등락률",
                 marker="o", linestyle="--", label="등락률 변화(Δ)")

    ax1.axvline(top_date, color="red", linestyle="--", label="이슈 발생일")

    ax1.set_xlabel("날짜")
    ax1.set_ylabel("수익률(%)")
    ax1.set_ylim(Y_MIN, Y_MAX)   # ⭐⭐ 여기가 Y축 고정 ⭐⭐
    ax1.tick_params(axis='x', rotation=45)

    # 거래량(log) 오른쪽 축
    ax2 = ax1.twinx()
    sns.lineplot(ax=ax2, data=window, x="date", y="거래량_log",
                 marker="o", color="gray", label="거래량(log)")
    ax2.set_ylabel("거래량 (log scale)")

    # 범례 합치기
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left")

    ax2.get_legend().remove()

    plt.title(f"[{stock}] Top 이벤트 ±7거래일 반응 (수익률/거래량)")
    plt.tight_layout()
    plt.show()

    print(f"===== [{stock}] 그래프 완료 =====")



# 실행
plot_event_reaction("LG에너지솔루션") 
plot_event_reaction("NAVER") 
plot_event_reaction("삼성전자") 
plot_event_reaction("셀트리온") 
plot_event_reaction("에코프로") 
plot_event_reaction("카카오") 
plot_event_reaction("펄어비스") 
plot_event_reaction("포스코퓨처엠") 
plot_event_reaction("한화오션") 
plot_event_reaction("현대차")


이슈강도 TOP 당일 뉴스기사

<LG에너지솔루션>

2025-09-08 총 633건 - [한국경제] 『[다산칼럼] 트럼프 정부가 간과하고 있는 것』 미국 조지아주 현대자동차-LG에너지솔루션 합작 배터리 공장 건설 현장에서 한국인 300여 명을 포함해 사상 초유의 집단 체포·구금 사태가 벌어졌다

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100601.20250908000942001

<NAVER>

2025-11-27 총 496건 - [일요신문] 『네이버·두나무 빅딜…‘운명의 칼자루’ 쥔 카카오 김범수의 선택은?』 네이버의 미래가 경쟁사인 카카오에 좌우되는 상황이 벌어졌다. 네이버파이낸셜과 두나무의 포괄적 주식교환(합병)의 성패가 김범수 카카오 미래이니셔티브센터장 결정에 달렸다는 분석이 나오는 것. 

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=01101202.20251127110254001

[한국경제] 『"이젠 업비트 악재도 걱정해야 하나"…네이버 주주의 한탄 [종목+]』 두나무가 네이버파이낸셜과 합병을 발표한 27일 업비트에서 445억원 규모의 해킹 사고가 발생하며 네이버 주가가 4% 넘게 하락했다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100601.20251127220425001

<삼성전자>

2025-10-31 총 933건 - [] 『젠슨 황 "제 치맥 동료들"…李 "저도 그자리 있었어야"』 31일 이재명 대통령과 젠슨 황 엔비디아 최고경영자(CEO)의 접견은 전날 젠슨 황과 재계 총수들의 ‘깐부치킨 회동’ 분위기를 이어가듯 화기애애하게 진행됐다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100601.20251031233112001

<셀트리온>

2025-09-23 총 106건 - [세계일보] 『셀트리온, 美관세 리스크 완전 해소...릴리 공장 4600억에 인수』 셀트리온이 글로벌 빅파마 일라이 릴리의 미국 바이오의약품 생산공장 인수를 위한 본계약을 맺으며 미국발 관세 리스크를 완전히 해소했다. 인수 금액은 약 4600억원 규모로, 향후 증설까지 총 1조4000억원을 투자할 계획이다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100101.20250923185002001

<에코프로>
2025-10-22 총 62건 - [중도일보] 『에코프로 창립 27주년…"국가대표 글로벌 기업 도약하자"』 에코프로가 창립 27주년을 맞아 창업 초기의 열정과 도전 정신을 바탕으로 국가 대표 글로벌 기업으로 도약하자는 의지를 다졌다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=01400351.20251022220541002

<카카오>
2025-02-04 총 471건 - [EBN] 『이재용·올트먼·손정의 '3國 AI 동맹'...삼성 '스타게이트' 합류』 챗GPT 개발사 오픈AI의 창업자 샘 올트먼 최고경영자(CEO)가 이재용 삼성전자 회장과 만나 글로벌 인공지능(AI) 인프라 프로젝트 '스타게이트' 협력을 밝혔다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=04100958.20250204234148001

<펄어비스>

2025-08-13 총 94건 - [매일경제] 『적자 커진 펄어비스, 하루새 24% 급락』 기존 게임의 매출이 부진한 탓에 올해 펄어비스의 적자 규모가 전년보다 커졌다. 글로벌 기대작인 '붉은사막' 출시도 연기하면서 주가가 하루 만에 24.17% 급락하기도 했다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100101.20250813180016002

<포스코퓨처엠>

2025-10-27 총 건 - [중도일보] 『포스코홀딩스 3분기 영업이익 6390억원… 작년比 13.5% 감소』포스코홀딩스가 올해 3분기 연결 기준 매출 17조 2610억원, 영업이익 6390억원을 기록했다고 27일 공시했다. 작년 동기 대비 매출은 5.8%, 영업이익은 13.5% 감소했다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=01400351.20251027220131002

<한화오션>

2025-10-30 총 316건 - [세계일보] 『[설왕설래] 핵추진잠수함』트럼프 대통령은 “한국은 미 필라델피아 조선소(필리조선소)에서 핵잠수함을 건조할 예정”이라고 밝혔다. 필리조선소는 국내 최다 잠수함 수주실적을 자랑하는 한화오션이 지난해 12월 인수한 자회사다

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=01100701.20251030232908003

<현대차>

2025-10-30 총 742건 - [한국경제] 『현대차가 점찍은 스트라드비젼, 코스닥 상장 예심 청구』 월 말 기준 최대주주는 지분 41.30%를 보유한 미국 전장 기술업체 앱티브다. 앱티브는 현대자동차그룹과 자율주행 기술 개발을 위한 합작법인 모셔널을 운영하는 곳이다. 김준환 스트라드비젼 대표는 11.74%를 보유한 2대 주주다.

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100601.20251030094445001

[메트로경제] 『이재용·정의선·젠슨황 '러브샷'...'AI 깐부 회동'에 삼성전자·현대차 주가↑』 젠슨 황 엔비디아 최고경영자(CEO)와 이재용 삼성전자 회장, 정의선 현대차그룹 회장이 30일 서울 강남구 코엑스에서 열린 엔비디아 지포스 게이머 페스티벌에서 참관객들에게 선물을 나눠주고 있다

https://www.bigkinds.or.kr/v2/news/newsDetailView.do?newsId=02100401.20251030235944001


# 발표 이외에 추가적인 분석

In [ ]:
# 이슈 발생일 기준 +3일 누적 수익률 비교 그래프
# 누적수익률(D+n)=(1+r0​)(1+r1​)(1+r2​)...(1+rn​)−1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------
# 1) 종목별 이슈강도 Top1 날짜 뽑기
# ----------------------------------------------------
top1 = df_all.loc[
    df_all.groupby("종목")["이슈강도"].idxmax()
].copy()

top1 = top1.sort_values("종목")

# ----------------------------------------------------
# 2) 각 종목에 대해 D0~D3 등락률 + 누적수익률 계산
# ----------------------------------------------------
results = []

for _, row in top1.iterrows():
    stock = row["종목"]
    event_date = row["date"]

    # 해당 종목만 정렬하고 인덱스 리셋
    df_stock = (
        df_all[df_all["종목"] == stock]
        .sort_values("date")
        .reset_index(drop=True)
    )

    # date 컬럼에서 event_date 위치 찾기
    pos = df_stock.index[df_stock["date"] == event_date]

    if len(pos) == 0:
        print(f"[경고] {stock} - {event_date} 날짜를 찾지 못했습니다.")
        continue

    idx = pos[0]

    # D0~D3 등락률 수집 (없으면 NaN)
    r_vals = []
    for offset in range(4):  # 0,1,2,3
        if idx + offset < len(df_stock):
            r_vals.append(df_stock.loc[idx + offset, "등락률"])
        else:
            r_vals.append(np.nan)

    r0, r1, r2, r3 = r_vals

    # 누적 수익률 계산 (NaN 제외하고 곱셈)
    valid_rs = [r for r in r_vals if pd.notnull(r)]
    daily_returns = [(1 + r/100) for r in valid_rs]
    cum_return = np.prod(daily_returns) - 1   # 소수 (예: 0.12 = 12%)

    results.append(
        [stock, event_date, row["이슈강도"], r0, r1, r2, r3, cum_return]
    )

cols = ["종목", "이슈 Top1 날짜", "이슈강도", "D0(당일)", "D1(+1일)", "D2(+2일)", "D3(+3일)", "누적수익률"]
df_cum = pd.DataFrame(results, columns=cols)

print(df_cum)

# ----------------------------------------------------
# 3) +3일 누적 수익률 바 차트
# ----------------------------------------------------
plt.figure(figsize=(12, 6))
plt.bar(df_cum["종목"], df_cum["누적수익률"] * 100,
        edgecolor="black")

plt.axhline(0, color="red", linestyle="--")
plt.title("이슈 발생일 기준 +3일 누적 수익률", fontsize=16)
plt.ylabel("누적 수익률 (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



이슈 발생일 기준 +3일 누적 수익률 인사이트

1.종목별로 이슈 발생 후 +3일 수익률은 극단적으로 양극화되는 패턴을 보이고 있다.

2.에코프로·현대차·삼성전자는 이슈가 단기 상승 모멘텀으로 작용합니다.

3.반대로 펄어비스·Naver·포스코퓨처엠은 이슈가 악재로 해석되어 하락으로 이어졌습니다.

4.같은 강도의 뉴스라도 시장 반응은 종목별로 크게 달라 일관되지 않았다.

5.단순 기사량이 아니라 이슈의 성격과 투자 심리가 단기 수익률을 결정하는 핵심 요인으로 나타났습니다.


In [ ]:
# 2) 이슈강도 vs 다음날 등락률
plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df_all_final,
    x='이슈강도',
    y='등락률_next',
    alpha=0.35
)

plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title("이슈강도 vs 다음 거래일 등락률", fontsize=14)
plt.xlabel("이슈강도", fontsize=12)
plt.ylabel("등락률_next (%)", fontsize=12)
plt.grid(alpha=0.2)

plt.show()

이슈강도 vs 다음 거래일 등락률 인사이트

1.이슈강도가 높아져도 다음날 등락률이 뚜렷하게 상승하거나 하락하는 경향은 나타나지 않았습니다.

2.대부분의 다음날 등락률은 -5% ~+5% 사이에 머물며, 이슈 강도와의 직접적 상관적은 매우 낮다.

3.일부 극단적 변동(급등·급락) 사례는 존재하지만, 이는 이슈강도보다는 개별 종목 고유 이벤트 영향으로 보인다.

4.결과적으로 뉴스량·검색량이 많다고 해서 다음날 주가가 크게 움직인다는 보장은 없다는 점이 확인되었다.

5.단기 예측력 관점에서 이슈강도 단일 변수로는 주가 방향성을 설명하기 어렵다는 결론을 도출할 수 있다.

In [ ]:
# 종목별 평균 이슈강도 계산
issue_mean = (
    df_all_final
    .groupby('종목')['이슈강도']
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12,6))
sns.barplot(
    x=issue_mean.index,
    y=issue_mean.values,
    palette="viridis"
)

plt.xticks(rotation=45)
plt.title("종목별 평균 이슈강도(뉴스건수 + 검색량)", fontsize=15)
plt.ylabel("평균 이슈강도", fontsize=12)
plt.xlabel("종목", fontsize=12)
plt.tight_layout()
plt.show()


종목별 평균 이슈강도 분석 인사이트

1.삼성전자는 모든 종목 중 가장 높은 평균 이슈강도를 기록하였으며, 시장·언론 관심도가 압도적으로 높습니다.

2.그 뒤를 카카오, LG에너지솔루션, 현대차 등이 이어지며 지속적인 뉴스·검색 노출이 많았던 종목군임을 확인할 수 있습니다.

3.에코프로·포스코퓨처엠·셀트리온 등은 중간 수준의 이슈강도를 보이며 이슈가 특정 시점에 집중되는 경향이 있습니다.

4.펄어비스는 전체 종목 중 가장 낮은 평균 이슈강도를 보이며, 평소 시장의 주목도가 상대적으로 낮은 종목으로 나타났습니다.

In [ ]:
# 종목별 상관관계 계산
corr_by_stock = (
    df_all_final.groupby('종목')[['이슈강도', '등락률_next']]
    .corr()
    .unstack()
    .iloc[:, 1]   # (이슈강도 → 등락률_next) 상관값만 추출
    .sort_values()
)

# 그래프
plt.figure(figsize=(12,6))
sns.barplot(
    x=corr_by_stock.index,
    y=corr_by_stock.values,
    palette="coolwarm"
)

plt.xticks(rotation=45)
plt.title("종목별 이슈강도 ↔ 다음날 등락률 상관관계", fontsize=15)
plt.ylabel("상관계수", fontsize=12)
plt.xlabel("종목", fontsize=12)
plt.axhline(0, color='black', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()


종목별 이슈강도 <-> 다음날 등락률 상관관계 인사이트

1.전체적으로 이슈강도와 다음날 주가 흐름 간의 상관성은 미약하며, 종목별 차이가 크게 나타났습니다.

2.Naver, 카카오처럼 상관계수가 음(-)인 종목은, 이슈가 많을수록 다음날 주가가 하락하는 경향이 나타났습니다.

3.반대로 에코프로·셀트리온·LG에너지솔루션은 상관계수가 양(+)으로 이슈가 늘어날수록 긍정적 가격 반응 가능성이 높았습니다.

4.그러나 모든 종목의 상관계수 절대값이 크지 않아, 이슈강도만으로 다음날 수익률을 예측하는 것은 제한적임을 확인할 수 있었습니다.

In [ ]:
# 4) 종목별 이슈강도 ↔ 다음날 거래량_log 상관관계
corr_volume = (
    df_all_final
    .groupby("종목")
    .apply(lambda x: x['이슈강도'].corr(x['거래량_log_next']))
    .reset_index(name="상관계수")
)

corr_volume = corr_volume.sort_values("상관계수", ascending=False)
print(corr_volume)

plt.figure(figsize=(14,6))
sns.barplot(data=corr_volume, x="종목", y="상관계수", palette="coolwarm")
plt.title("종목별 이슈강도 ↔ 다음날 거래량 변화 상관관계", fontsize=16)
plt.axhline(0, color="black", linestyle="--")
plt.xticks(rotation=45)
plt.show()

종목별 이슈강도 <-> 다음날 거래량 변화 상관관계

1.전 종목에서 상관계수가 모두 양(+)으로 나타나, 이슈가 강할수록 다음날 거래량이 증가하는 경향이 공통적으로 확인되었습니다.

2.특히 에코프로, 포스코퓨처엠, Naver, 펄어비스는 상관계수가 높아, 이슈 발생 시 투자자 관심이 즉각적으로 거래량 증가로 이어지는 종목으로 분류됩니다,

3.반면 삼성전자,현대오션 등 대형주는 상관계수가 상대적으로 낮아, 이슈에 대한 거래량 반응이 크지 않은 안정적 패턴을 보였다.

4.전체적으로 거래량은 등략률보다 이슈강도와의 관계가 더 선명하여, 이슈 발생 -> 투자 주목도 증가 -> 거래량 확대라는 흐름이 확인된다.

In [ ]:
# 1) 전체 상관관계 히트맵
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.heatmap(
    df_all_final[['이슈강도','검색량','뉴스건수','등락률_next','거래량_log_next']].corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title("전체 상관관계 Heatmap")
plt.show()


전체 상관관계 Heatmap 상관관계

1.이슈강도는 검색량(0.74) 및 뉴스건수(0.71) 와 매우 높은 상관관계를 보여, 세 변수가 함께 증가하는 구조임을 확인할 수 있었습니다.

2.이슈강도와 등락률_next의 상관계수는 거의 0(0.03) 으로, 이슈의 크기가 다음날 주가 방향을 설명해주지 못한다는 점을 보여주였습니다.

3.반면 거래량_log_next는 이슈강도와 중간 정도의 양(+) 상관(0.56) 을 보이며, 이슈가 커질수록 거래량이 증가하는 경향이 존재하였습니다.

4.전체적으로 이슈는 가격보다는 거래량(투자자 관심도)을 움직이는 경향이 더 강하다는 사실이 드러났습니다.